# Bronze Layer - CDC Event Ingestion

This notebook ingests raw CDC events from Kafka into Delta tables with **schema drift detection** and monitoring.

In [ ]:
%run ../helpers/NB_catalog_helpers

In [ ]:
%run ../helpers/NB_schema_drift_helpers

In [ ]:
%run ../helpers/NB_schema_contracts

In [ ]:
from pyspark.sql.functions import col, current_timestamp, regexp_extract
from pyspark.sql.types import StructType, StructField, StringType, LongType
import uuid

dbutils.widgets.text("LANDING_PATH",     "/Volumes/workspace/default/mnt/bronze-landing")
dbutils.widgets.text("CATALOG",          "workspace")
dbutils.widgets.text("BRONZE_SCHEMA",    "bronze")
dbutils.widgets.text("CHECKPOINT_PATH",  "/Volumes/workspace/default/mnt/checkpoints/bronze_cdc")
dbutils.widgets.text("CHECKPOINT_ROOT",  "/Volumes/workspace/default/mnt/checkpoints")
dbutils.widgets.text("SCHEMA_LOCATION",  "/Volumes/workspace/default/mnt/checkpoints/bronze_autoloader_schema")
dbutils.widgets.text("MONITORING_SCHEMA", "monitoring")
dbutils.widgets.text("SCHEMA_POLICY",    "additive_only")
dbutils.widgets.text("ALERT_CHANNEL",    "log")
dbutils.widgets.text("WEBHOOK_URL",      "")

CONFIG = {
    "landing_path":      dbutils.widgets.get("LANDING_PATH"),
    "catalog":           dbutils.widgets.get("CATALOG"),
    "bronze_schema":     dbutils.widgets.get("BRONZE_SCHEMA"),
    "checkpoint_path":   dbutils.widgets.get("CHECKPOINT_PATH"),
    "schema_location":   dbutils.widgets.get("SCHEMA_LOCATION"),
    "monitoring_schema": dbutils.widgets.get("MONITORING_SCHEMA"),
    "schema_policy":     dbutils.widgets.get("SCHEMA_POLICY"),
    "alert_channel":     dbutils.widgets.get("ALERT_CHANNEL"),
    "webhook_url":       dbutils.widgets.get("WEBHOOK_URL"),
}

PIPELINE_RUN_ID = str(uuid.uuid4())
drift_table_fqn = f"{CONFIG['catalog']}.{CONFIG['monitoring_schema']}.schema_drift_log"

print(f"Landing path    : {CONFIG['landing_path']}")
print(f"Schema location : {CONFIG['schema_location']}")

In [ ]:
# Initialize monitoring table for schema drift tracking
create_schema_drift_table(spark, CONFIG['catalog'], CONFIG['monitoring_schema'])

# Initialize schema
ensure_schema_exists(CONFIG['catalog'], CONFIG['bronze_schema'])

# Task 2.1 — ensure quarantine table exists
ensure_bronze_quarantine(spark, CONFIG['catalog'])

In [ ]:
# NDJSON schema written by kafka_to_volume.py — one record per Kafka message.
# partition is IntegerType to match Bronze Delta tables created by the old Kafka source.
from pyspark.sql.types import IntegerType as _IntegerType

_LANDING_SCHEMA = StructType([
    StructField("topic",       StringType(),    True),
    StructField("partition",   _IntegerType(),  True),
    StructField("offset",      LongType(),      True),
    StructField("timestamp",   LongType(),      True),   # ms epoch from Kafka
    StructField("message_key", StringType(),    True),
    StructField("value",       StringType(),    True),
])

bronze_events = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format",         "json")
    .option("cloudFiles.schemaLocation", CONFIG["schema_location"])
    .schema(_LANDING_SCHEMA)
    .load(CONFIG["landing_path"])
    # Normalise timestamp: ms epoch → Spark TimestampType (matches old kafka_timestamp column)
    .withColumn("kafka_timestamp", (col("timestamp") / 1000).cast("timestamp"))
    .withColumn("ingested_at",     current_timestamp())
    # Extract schema / table from topic: cdc.{schema}.{table}
    .withColumn("source_schema",   regexp_extract(col("topic"), r"^[^.]+\.([^.]+)\.[^.]+$", 1))
    .withColumn("table_name",      regexp_extract(col("topic"), r"^[^.]+\.[^.]+\.([^.]+)$", 1))
    .filter(col("table_name") != "")
    .drop("timestamp")   # replaced by kafka_timestamp above
)

In [ ]:
def validate_bronze_table_schema(table_name, table_df):
    target_table = f"{CONFIG['catalog']}.{CONFIG['bronze_schema']}.{table_name}"
    contract_key = f"bronze.{table_name}"

    if contract_key not in SCHEMA_CONTRACTS:
        print(f"  ⚠️ No schema contract found for {contract_key}")
        return

    expected_schema = get_schema_contract(contract_key)
    actual_schema = get_existing_schema(spark, target_table) or table_df.schema
    print(f"Validating schema for {target_table} against {contract_key}")

    try:
        is_valid, drift_result = validate_schema_with_policy(
            spark,
            expected_schema=expected_schema,
            actual_schema=actual_schema,
            table_name=target_table,
            drift_table=drift_table_fqn,
            source_system="postgres_cdc",
            pipeline_run_id=PIPELINE_RUN_ID,
            policy=CONFIG['schema_policy'],
            alert_channel=CONFIG['alert_channel'],
            webhook_url=CONFIG['webhook_url'] if CONFIG['webhook_url'] else None
        )
        if drift_result['has_drift']:
            print(f"  ⚠️ Schema drift detected: {drift_result['severity']}")
        else:
            print("  ✅ Schema validation passed")
    except SchemaDriftException as e:
        raise RuntimeError(f"Schema drift blocked pipeline for {table_name}: {e}") from e

In [ ]:
import json as _json

def _validate_envelope(value: str | None, kafka_timestamp) -> str | None:
    """
    Validate a raw Debezium envelope before writing to Bronze.
    Returns None if valid, or a short reason string for quarantine routing.
    """
    if value is None or value.strip() == "":
        return "null_or_empty_value"
    try:
        _json.loads(value)
    except ValueError:
        return "invalid_json"
    return None

In [ ]:
def write_bronze(batch_df, batch_id):
    """
    Write batch to Bronze tables with:
    - Envelope validation + quarantine routing (Task 2.1)
    - Schema drift detection
    - 30-day TTL properties on each table (Task 2.2)
    - Volume anomaly detection per table (Task 4.4)
    """
    if not batch_df.take(1):
        return

    quarantine_fqn = f"{CONFIG['catalog']}.{CONFIG['bronze_schema']}.quarantine"
    now = datetime.now(timezone.utc)

    # Collect and validate row-by-row
    rows = batch_df.collect()
    valid_rows   = []
    invalid_rows = []

    for row in rows:
        reason = _validate_envelope(row["value"], row["kafka_timestamp"])
        if reason:
            invalid_rows.append((row, reason))
        else:
            valid_rows.append(row)

    # ------------------------------------------------------------------ #
    # Route invalid rows → quarantine                                      #
    # ------------------------------------------------------------------ #
    if invalid_rows:
        from pyspark.sql import Row as _Row
        q_rows = [
            _Row(
                topic             = r.topic,
                partition         = r.partition,
                offset            = r.offset,
                kafka_timestamp   = r.kafka_timestamp,
                message_key       = r.message_key,
                value             = r.value,
                ingested_at       = r.ingested_at,
                source_schema     = r.source_schema,
                table_name        = r.table_name,
                quarantine_reason = reason,
                quarantined_at    = now,
            )
            for r, reason in invalid_rows
        ]
        (
            spark.createDataFrame(q_rows)
            .write.format("delta")
            .mode("append")
            .saveAsTable(quarantine_fqn)
        )
        print(f"Quarantined {len(invalid_rows)} row(s) → {quarantine_fqn}")

    if not valid_rows:
        return

    valid_df = spark.createDataFrame(valid_rows, schema=batch_df.schema)

    # ------------------------------------------------------------------ #
    # Write valid rows by table                                            #
    # ------------------------------------------------------------------ #
    tables = [row.table_name for row in valid_df.select("table_name").distinct().collect()]

    for table_name in tables:
        target = f"{CONFIG['catalog']}.{CONFIG['bronze_schema']}.{table_name}"

        table_df = valid_df.filter(col("table_name") == table_name)
        row_count = table_df.count()

        # Schema drift validation
        validate_bronze_table_schema(table_name, table_df)

        # Append to Delta table
        (
            table_df
            .write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(target)
        )

        # Task 2.2 — apply 30-day TTL (idempotent)
        set_bronze_ttl(spark, target, retain_days=30)

        # Task 4.4 — volume anomaly detection
        vol_status = check_volume_anomaly(
            spark, run_id=PIPELINE_RUN_ID,
            table_name=table_name, current_count=row_count,
            catalog=CONFIG["catalog"],
        )
        if vol_status == "WARN":
            alert_dq_failure(
                table_name=table_name, check_name="batch_row_count",
                observed_value=row_count, layer="bronze",
                alert_channel=CONFIG["alert_channel"],
                webhook_url=CONFIG["webhook_url"] if CONFIG["webhook_url"] else None,
            )

        print(f"Wrote {row_count} valid records to {target} [volume={vol_status}]")

print("Bronze streaming query configured")

In [ ]:
# Start streaming write
query = (
    bronze_events.writeStream
    .foreachBatch(write_bronze)
    .option("checkpointLocation", CONFIG["checkpoint_path"])
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .start()
)

# Wait for completion
query.awaitTermination()

print(f"✅ Bronze ingestion completed. Run ID: {PIPELINE_RUN_ID}")